# SQL Database Project — Synthetic E-Commerce Database

**Reproducible Google Colab notebook | SQLite + Python standard library**

This notebook generates a fully original synthetic e-commerce database. No dataset is downloaded, scraped, or read from an external source. **It is fictional data, not real customer or business data.**

**Assignment mapping:** 6 related tables, at least 7 columns, a 1,500-row `orders` table, nominal + ordinal + interval + ratio data, foreign keys, a compound primary key, randomized realistic records, constraints, and a validation section.

Run **Runtime → Run all**. The notebook generates `Ecommerce_Database.db` in the current working directory. The optional final cell downloads the SQLite file for submission.

## 1. Design decisions and measurement scales

| Measurement scale | Example column | Why |
|---|---|---|
| Nominal | `customers.region` | Unordered category |
| Ordinal | `customers.satisfaction_level` | Ordered labels: Low < Medium < High < Very High |
| Interval | `products.storage_temperature_c` | Degrees Celsius have an arbitrary zero; differences are meaningful, ratios are not |
| Ratio | `order_items.quantity`, `products.unit_price_cents` | Counts and monetary amounts have meaningful zero |

**Important:** Measurement scales are conceptual; SQLite types (`INTEGER`, `REAL`, `TEXT`) describe storage. Temperatures are a synthetic warehouse storage observation assigned to product records, not a validated product requirement.

**Rationale for multiple tables:** Separating categories, customers, products, orders, line items, and payments reduces repeated data and supports one-to-many relationships. `order_items` uses the **compound primary key** (`order_id`, `line_number`), while `products.category_id`, `orders.customer_id`, `order_items.order_id`, and related fields are foreign keys. Prices are stored as integer cents to avoid floating-point rounding errors.

**Generation policy:** All records come from deterministic Python pseudorandom sampling (`seed = 20261106`), custom fictional names, and reserved `example.com` email addresses. No actual person is represented. There are no third-party datasets.

## 2. Generate the database
The following code defines the complete SQL schema, creates realistic linked records, enforces integrity constraints, and provides automatic validation.

In [ ]:
"""Generate an original, reproducible, entirely synthetic SQLite ecommerce database.

No datasets are downloaded or loaded. Requires only the Python standard library.
"""
from __future__ import annotations
import datetime as dt
import random
import sqlite3
from pathlib import Path

SEED = 20261106
COUNTS = {'categories': 10, 'products': 150, 'customers': 500, 'orders': 1500}
CATEGORY_NAMES = [
    'Automotive Lighting', 'Exterior Styling', 'Interior Accessories', 'Electronics',
    'Maintenance', 'Wheels and Tyres', 'Performance Parts', 'Safety Equipment',
    'Tools', 'Lifestyle Accessories'
]
FIRST_NAMES = ['Avery', 'Jordan', 'Taylor', 'Morgan', 'Riley', 'Casey', 'Quinn', 'Harper',
               'Parker', 'Alex', 'Jamie', 'Skyler', 'Drew', 'Rowan', 'Emery', 'Sage']
LAST_NAMES = ['Reed', 'Brooks', 'Ellis', 'Rivera', 'Chen', 'Patel', 'Morris', 'Kim',
              'Hayes', 'Lewis', 'Cole', 'Shah', 'Martin', 'Stone', 'Wright', 'Nguyen']
PRODUCT_NOUNS = ['Kit', 'Module', 'Cover', 'Adapter', 'Assembly', 'Insert', 'Bracket', 'Set',
                 'Panel', 'Mount', 'Trim', 'Controller', 'Harness', 'Package', 'Accessory']
FINISHES = ['Matte Black', 'Gloss Black', 'Silver', 'Carbon Look', 'Clear', 'Smoked', 'White']
REGIONS = ['North', 'South', 'East', 'West', 'Central']


def create_database(db_path: str | Path = 'Ecommerce_Database.db') -> Path:
    """Create database from scratch. Same seed => same records and values."""
    path = Path(db_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()  # Explicit rebuild avoids duplicate rows on re-run.
    rng = random.Random(SEED)
    con = sqlite3.connect(str(path))
    con.execute('PRAGMA foreign_keys = ON')
    try:
        con.executescript('''
        CREATE TABLE categories (
            category_id INTEGER PRIMARY KEY,
            category_name TEXT NOT NULL UNIQUE,
            display_priority INTEGER NOT NULL CHECK (display_priority BETWEEN 1 AND 10)
        );
        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            display_name TEXT NOT NULL,
            synthetic_email TEXT NOT NULL UNIQUE CHECK(synthetic_email LIKE '%@example.com'),
            region TEXT NOT NULL CHECK(region IN ('North','South','East','West','Central')),
            joined_date TEXT NOT NULL CHECK (joined_date GLOB '????-??-??'),
            loyalty_tier TEXT NOT NULL CHECK(loyalty_tier IN ('Bronze','Silver','Gold','Platinum')),
            satisfaction_level TEXT NOT NULL CHECK(satisfaction_level IN ('Low','Medium','High','Very High'))
        );
        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            category_id INTEGER NOT NULL REFERENCES categories(category_id),
            product_name TEXT NOT NULL UNIQUE,
            unit_price_cents INTEGER NOT NULL CHECK(unit_price_cents > 0),
            unit_cost_cents INTEGER NOT NULL CHECK(unit_cost_cents > 0 AND unit_cost_cents < unit_price_cents),
            available_stock INTEGER NOT NULL CHECK(available_stock >= 0),
            storage_temperature_c REAL NOT NULL CHECK(storage_temperature_c BETWEEN -10 AND 45),
            popularity_rank INTEGER NOT NULL CHECK(popularity_rank BETWEEN 1 AND 5)
        );
        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer_id INTEGER NOT NULL REFERENCES customers(customer_id),
            order_date TEXT NOT NULL CHECK(order_date GLOB '????-??-??'),
            order_status TEXT NOT NULL CHECK(order_status IN ('Pending','Shipped','Delivered','Cancelled')),
            shipping_region TEXT NOT NULL CHECK(shipping_region IN ('North','South','East','West','Central')),
            shipping_cents INTEGER NOT NULL CHECK(shipping_cents >= 0),
            order_total_cents INTEGER NOT NULL CHECK(order_total_cents >= 0)
        );
        CREATE TABLE order_items (
            order_id INTEGER NOT NULL REFERENCES orders(order_id),
            line_number INTEGER NOT NULL CHECK(line_number > 0),
            product_id INTEGER NOT NULL REFERENCES products(product_id),
            quantity INTEGER NOT NULL CHECK(quantity BETWEEN 1 AND 5),
            unit_price_cents INTEGER NOT NULL CHECK(unit_price_cents > 0),
            PRIMARY KEY (order_id, line_number)
        );
        CREATE TABLE payments (
            payment_id INTEGER PRIMARY KEY,
            order_id INTEGER NOT NULL UNIQUE REFERENCES orders(order_id),
            method TEXT NOT NULL CHECK(method IN ('Card','Wallet','Bank Transfer')),
            payment_status TEXT NOT NULL CHECK(payment_status IN ('Paid','Pending','Refunded')),
            amount_cents INTEGER NOT NULL CHECK(amount_cents >= 0),
            paid_date TEXT CHECK(paid_date IS NULL OR paid_date GLOB '????-??-??')
        );
        CREATE INDEX idx_products_category ON products(category_id);
        CREATE INDEX idx_orders_customer ON orders(customer_id);
        CREATE INDEX idx_order_items_product ON order_items(product_id);
        ''')
        con.executemany('INSERT INTO categories VALUES (?, ?, ?)',
                        [(i, name, i) for i, name in enumerate(CATEGORY_NAMES, 1)])

        start = dt.date(2024, 1, 1)
        end = dt.date(2026, 10, 1)
        def date_between(a: dt.date, b: dt.date) -> dt.date:
            return a + dt.timedelta(days=rng.randint(0, (b-a).days))

        customers = []
        joined_dates = {}
        regions = {}
        for cid in range(1, COUNTS['customers'] + 1):
            joined = date_between(start, dt.date(2025, 12, 31))
            region = rng.choice(REGIONS)
            joined_dates[cid] = joined
            regions[cid] = region
            customers.append((cid, f'{rng.choice(FIRST_NAMES)} {rng.choice(LAST_NAMES)}',
                              f'synthetic.customer{cid:04d}@example.com', region,
                              joined.isoformat(), rng.choices(['Bronze','Silver','Gold','Platinum'], [50,28,17,5])[0],
                              rng.choices(['Low','Medium','High','Very High'],[7,20,51,22])[0]))
        con.executemany('INSERT INTO customers VALUES (?, ?, ?, ?, ?, ?, ?)', customers)

        products = []
        prices = {}
        for pid in range(1, COUNTS['products'] + 1):
            cat_id = (pid - 1) % 10 + 1
            price = rng.randint(1500, 85000)
            cost = round(price * rng.uniform(.40, .78))
            prices[pid] = price
            products.append((pid, cat_id, f'{FINISHES[(pid-1)%len(FINISHES)]} {CATEGORY_NAMES[cat_id-1]} {PRODUCT_NOUNS[(pid-1)//10 % len(PRODUCT_NOUNS)]} P{pid:03}',
                             price, cost, rng.randint(0, 250), round(rng.uniform(10, 34), 1), rng.randint(1, 5)))
        con.executemany('INSERT INTO products VALUES (?, ?, ?, ?, ?, ?, ?, ?)', products)

        orders, items, payments = [], [], []
        for oid in range(1, COUNTS['orders'] + 1):
            cid = rng.randint(1, COUNTS['customers'])
            day = date_between(joined_dates[cid], end)
            status = rng.choices(['Delivered','Shipped','Pending','Cancelled'], [72,13,10,5])[0]
            shipping = 0 if status == 'Cancelled' else rng.choice([0, 500, 1000, 1500, 2000])
            n_items = rng.randint(1, 4)
            selected_products = rng.sample(range(1, COUNTS['products']+1), n_items)
            subtotal = 0
            for line_no, pid in enumerate(selected_products, 1):
                qty = rng.randint(1, 5)
                unit_price = prices[pid]  # Historical sale price snapshot in cents.
                subtotal += qty * unit_price
                items.append((oid, line_no, pid, qty, unit_price))
            total = 0 if status == 'Cancelled' else subtotal + shipping
            # Cancelled order items remain for audit, but are not charged.
            orders.append((oid, cid, day.isoformat(), status, regions[cid], shipping, total))
            pay_status = 'Refunded' if status == 'Cancelled' else ('Pending' if status == 'Pending' else 'Paid')
            amount = 0 if pay_status == 'Refunded' else total
            paid_day = day.isoformat() if pay_status == 'Paid' else None
            payments.append((oid, oid, rng.choices(['Card','Wallet','Bank Transfer'], [65,25,10])[0],
                             pay_status, amount, paid_day))
        con.executemany('INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?)', orders)
        con.executemany('INSERT INTO order_items VALUES (?, ?, ?, ?, ?)', items)
        con.executemany('INSERT INTO payments VALUES (?, ?, ?, ?, ?, ?)', payments)
        con.commit()
        return path
    finally:
        con.close()


def validate_database(db_path: str | Path) -> dict:
    """Assert referential integrity, data volume, all measurement scales, and monetary consistency."""
    con = sqlite3.connect(str(db_path))
    con.execute('PRAGMA foreign_keys=ON')
    try:
        tables = ['categories','customers','products','orders','order_items','payments']
        counts = {name: con.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0] for name in tables}
        assert counts['orders'] >= 1000, 'At least one table must contain 1000+ rows'
        assert counts['orders'] == counts['payments'] == COUNTS['orders']
        assert counts['order_items'] >= counts['orders']
        assert not con.execute('PRAGMA foreign_key_check').fetchall(), 'Foreign key violation'
        assert con.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
        assert not con.execute('''SELECT o.order_id FROM orders o
            JOIN (SELECT order_id, SUM(quantity * unit_price_cents) AS subtotal
                  FROM order_items GROUP BY order_id) s ON o.order_id=s.order_id
            WHERE (o.order_status <> 'Cancelled' AND o.order_total_cents <> s.subtotal + o.shipping_cents)
               OR (o.order_status = 'Cancelled' AND (o.order_total_cents <> 0 OR o.shipping_cents <> 0))''').fetchall()
        assert not con.execute('''SELECT 1 FROM orders o JOIN payments p ON o.order_id=p.order_id
            WHERE (o.order_status='Cancelled' AND (p.payment_status<>'Refunded' OR p.amount_cents<>0))
               OR (o.order_status='Pending' AND p.payment_status<>'Pending')
               OR (o.order_status IN ('Delivered','Shipped') AND p.payment_status<>'Paid')
               OR p.amount_cents<>o.order_total_cents''').fetchall()
        assert con.execute('SELECT COUNT(DISTINCT satisfaction_level) FROM customers').fetchone()[0] == 4
        assert con.execute('SELECT COUNT(DISTINCT storage_temperature_c) FROM products').fetchone()[0] > 1
        # Enforcement test: rejects invalid ratio quantities and unknown parent foreign keys.
        for sql in ["INSERT INTO order_items VALUES(1,999,1,-1,100)",
                    "INSERT INTO order_items VALUES(999999,1,1,1,100)"]:
            try:
                con.execute(sql)
                raise AssertionError('Constraint test unexpectedly passed')
            except sqlite3.IntegrityError:
                pass
        return counts
    finally:
        con.rollback()
        con.close()



## 3. Build and validate the SQLite file

In [ ]:
from pathlib import Path
DB_PATH = Path('Ecommerce_Database.db')
created_path = create_database(DB_PATH)
row_counts = validate_database(created_path)
print('SQLite file:', created_path.resolve())
print('Table row counts:')
for name, count in row_counts.items():
    print(f'  {name:15s} {count:,}')
print('TOTAL ROWS:', f'{sum(row_counts.values()):,}')
print('All validation assertions PASSED')

## 4. Explore the actual SQL schema

The SQL DDL is defined in the generator. This query extracts SQLite's stored `CREATE TABLE` statements, making it straightforward to include a textual schema in the PDF report.

In [ ]:
with sqlite3.connect(DB_PATH) as con:
    for name, ddl in con.execute("SELECT name, sql FROM sqlite_master WHERE type='table' ORDER BY name"):
        print('\n' + name.upper() + '\n' + ddl + ';')

## 5. Run example SQL queries
These queries demonstrate joining tables, grouping values, filtering records, and retrieving SQL data for downstream analysis.

In [ ]:
with sqlite3.connect(DB_PATH) as con:
    con.row_factory = sqlite3.Row
    examples = {
        'Orders by status': "SELECT order_status, COUNT(*) AS orders FROM orders GROUP BY order_status ORDER BY orders DESC",
        'Revenue by category (excluding cancelled)': """SELECT c.category_name,
               ROUND(SUM(oi.quantity * oi.unit_price_cents) / 100.0, 2) AS sales_value
               FROM order_items oi JOIN products p ON oi.product_id = p.product_id
               JOIN categories c ON p.category_id = c.category_id
               JOIN orders o ON o.order_id = oi.order_id
               WHERE o.order_status <> 'Cancelled'
               GROUP BY c.category_name ORDER BY sales_value DESC""",
        'Top five products by units sold': """SELECT p.product_name, SUM(oi.quantity) AS units
               FROM order_items oi JOIN orders o ON oi.order_id=o.order_id
               JOIN products p ON oi.product_id=p.product_id
               WHERE o.order_status <> 'Cancelled'
               GROUP BY p.product_id ORDER BY units DESC LIMIT 5""",
        'Customers by ordered satisfaction': """SELECT satisfaction_level, COUNT(*) AS n
               FROM customers GROUP BY satisfaction_level
               ORDER BY CASE satisfaction_level WHEN 'Low' THEN 1 WHEN 'Medium' THEN 2
                    WHEN 'High' THEN 3 ELSE 4 END"""
    }
    for label, sql in examples.items():
        print('\n###', label)
        for row in con.execute(sql).fetchall():
            print(tuple(row))

## 6. Validate constraints manually

The automated validator has already checked foreign keys, row counts, order totals, payment consistency, and selected bad data inserts. For additional evidence, the queries below show SQLite's integrity check and foreign-key check.

In [ ]:
with sqlite3.connect(DB_PATH) as con:
    con.execute('PRAGMA foreign_keys = ON')
    print('Integrity:', con.execute('PRAGMA integrity_check').fetchone()[0])
    print('Foreign key errors:', con.execute('PRAGMA foreign_key_check').fetchall())
    print('Compound key fields:', con.execute("PRAGMA table_info('order_items')").fetchall())

## 7. Ethical and privacy considerations

- Data is fully synthetic and created exclusively by this notebook, avoiding consent and licensing issues associated with reusing real customer datasets.
- Names are artificial combinations and emails end in the reserved `example.com` domain, so they are not intended to reach real people.
- No actual card numbers, payment tokens, phone numbers, addresses, or credentials are generated.
- Synthetic data still requires labeling: it should never be presented as an authentic business dataset or used as evidence of actual commercial performance.
- If this schema were used for real data, access controls, retention limits, encryption, and applicable privacy requirements would be needed.

**Limitation:** This is a simplified teaching database. Refunds are represented as zero amount on cancelled orders, not as real cash-flow movements, and inventory stock is a snapshot rather than a live ledger.

## 8. Download database (Google Colab only)
If using Colab, run the next cell to download the validated `.db` file. For local Jupyter, simply locate the file in the notebook working directory.

In [ ]:
try:
    from google.colab import files
    files.download(str(DB_PATH))
except ImportError:
    print('Not running on Google Colab. Database is saved at:', DB_PATH.resolve())